In [1]:
#!pip install openmeteo_requests
from datetime import datetime
import openmeteo_requests

class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, step: int = 10):
        self.current = current_speed
        self.max = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        new_speed = self.current + self.step
        if new_speed > self.max:
            raise StopIteration
        self.current = new_speed
        return new_speed


class DecreaseSpeed:
    def __init__(self, current_speed: int, min_speed: int = 0, step: int = 10):
        self.current = current_speed
        self.min = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        new_speed = self.current - self.step
        if new_speed < self.min:
            raise StopIteration
        self.current = new_speed
        return new_speed


class Car:
    _total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed: int = 0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        if current_speed > 0:
            self.on_road = True
            Car._total_cars_on_road += 1
        else:
            self.on_road = False

    def accelerate(self, upper_border=None, step=10):
        initial_speed = self.current_speed
        was_off_road = not self.on_road

        if upper_border is not None:
            if upper_border > self.max_speed:
                upper_border = self.max_speed
            if upper_border < self.current_speed:
                return
            if self.current_speed == upper_border:
                print(f"INFO: The speed of this car has been increased from {self.current_speed} to {self.current_speed}")
                return

            inc = IncreaseSpeed(self.current_speed, self.max_speed, step)
            for new_speed in inc:
                if new_speed <= upper_border:
                    self.current_speed = new_speed
                    print(f"INFO: Speed increases by {step}")
                else:
                    break
        else:
            inc = IncreaseSpeed(self.current_speed, self.max_speed, step)
            try:
                new_speed = next(inc)
                if new_speed <= self.max_speed:
                    self.current_speed = new_speed
                    print(f"INFO: Speed increases by {step}")
            except StopIteration:
                pass

        if was_off_road and self.current_speed > 0:
            self.on_road = True
            Car._total_cars_on_road += 1

        if self.current_speed != initial_speed or (upper_border is not None and self.current_speed == upper_border):
            print(f"INFO: The speed of this car has been increased from {initial_speed} to {self.current_speed}")

    def brake(self, lower_border=None, step=10):
        initial_speed = self.current_speed

        if lower_border is not None:
            if lower_border < 0:
                lower_border = 0
            if lower_border > self.current_speed:
                return
            if self.current_speed == lower_border:
                print(f"INFO: The speed of this car has been decreased from {self.current_speed} to {self.current_speed}")
                return

            dec = DecreaseSpeed(self.current_speed, 0, step)
            for new_speed in dec:
                if new_speed >= lower_border:
                    self.current_speed = new_speed
                    print(f"INFO: Speed decreases by {step}")
                else:
                    break
        else:
            dec = DecreaseSpeed(self.current_speed, 0, step)
            try:
                new_speed = next(dec)
                if new_speed >= 0:
                    self.current_speed = new_speed
                    print(f"INFO: Speed decreases by {step}")
            except StopIteration:
                pass

        if self.current_speed != initial_speed or (lower_border is not None and self.current_speed == lower_border):
            print(f"INFO: The speed of this car has been decreased from {initial_speed} to {self.current_speed}")

    def parking(self):
        if not self.on_road:
            return
        if self.current_speed != 0:
            self.brake(0)
        else:
            self.brake(0) 
        self.on_road = False
        Car._total_cars_on_road -= 1
        print("Parking the car...")

    @classmethod
    def total_cars(cls):
        return cls._total_cars_on_road

    @staticmethod
    def show_weather():
        openmeteo = openmeteo_requests.Client()
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow"
        }
        response = openmeteo.weather_api(url, params=params)[0]
        current = response.Current()
        current_temperature_2m = current.Variables(0).Value()
        current_apparent_temperature = current.Variables(1).Value()
        current_rain = current.Variables(2).Value()
        current_wind_speed_10m = current.Variables(3).Value()

        print(f"Current temperature: {round(current_temperature_2m, 0)} C")
        print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
        print(f"Current rain: {current_rain} mm")
        print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")


if __name__ == "__main__":
    car1 = Car(100, 20)  
    car2 = Car(60, 30)
    car3 = Car(100, 0)

    print(f"Total cars on road: {Car.total_cars()}\n")

    car1.accelerate(100)
    print()

    car2.accelerate(50)
    print()

    print("Speed of car 1:", car1.current_speed)
    print("Speed of car 2:", car2.current_speed)
    print()

    car1.brake(10)
    print()

    car2.brake(0)
    print("Total cars on road:", Car.total_cars())
    car2.parking()
    print("Total cars on road:", Car.total_cars())
    print()

    car3.accelerate(80)
    car3.show_weather()
    print("Total cars on road:", Car.total_cars())
    print()

    car2.accelerate(10)
    print("Total cars on road:", Car.total_cars())
    print()

    Car.show_weather()

ModuleNotFoundError: No module named 'openmeteo_requests'